# The pipeline step by step

This notebook walks through the Python API of mastpipe on a **simulated** laboratory
night, one stage per section, looking at the objects each stage produces. It needs no
data and runs in under a minute. The same functions, in the same order, are what
`mastpipe quicklook` runs; here they are called by hand so each intermediate result can be
inspected.

The night uses a small fiber layout made in this notebook — three telescopes, each with a
science and a sky fiber, plus three calibration fibers — because the layout is an input
table and that is the simplest way to see what it does.

In [ ]:
import tempfile
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits

from mastpipe.config.loader import build_effective
from mastpipe.instruments.registry import load_header_map, load_instrument, load_line_table
from mastpipe.sim.simulator import simulate_night

work = Path(tempfile.mkdtemp(prefix="mastpipe-tutorial-"))
night = work / "night"

instrument = load_instrument("highspec")
# a 12-px pitch: give the trace, line and extraction steps half-widths that fit it
effective = build_effective(
    instrument, "mgb518", "main",
    overrides=[{"trace": {"trace_halfwidth": 5}, "lines": {"halfwidth": 5}, "extract": {"halfwidth": 5.0}}],
)
line_table = load_line_table("highspec", "mgb518")
params = effective.params

rows = []
y = 40.0
for k, telescope in enumerate(["T01", "T02", "T03"]):
    rows.append(dict(trace=3 * k, role="arc", telescope=None, y_nominal=y, slope_nominal=0.002, aperture=None))
    rows.append(dict(trace=3 * k + 1, role="science", telescope=telescope, y_nominal=y + 12, slope_nominal=0.002, aperture=None))
    rows.append(dict(trace=3 * k + 2, role="sky", telescope=telescope, y_nominal=y + 24, slope_nominal=0.002, aperture=None))
    y += 36
layout = pd.DataFrame(rows)
layout

## A simulated night

`simulate_night` writes raw frames with the headers the HighSpec header map understands,
and a truth table. The full arc lights every trace; the calibration-fiber arc lights the
arc traces only; the science frames carry no arc light. Lines in the calibration arc and the
science spectra drift linearly in time from the full arc.

In [ ]:
truth = simulate_night(
    effective, layout, line_table, night,
    seed=1, n_science=2, n_cal_arc=1, cosmic_rays=10, fiber_preset="tutorial9",
)
truth.frames[["file", "exposure_type", "arc_kind", "mid_time", "shift_px"]]

In [ ]:
science_file = night / truth.frames.loc[truth.frames.exposure_type == "science", "file"].iloc[0]
data = fits.getdata(science_file).astype(float)
fig, ax = plt.subplots(figsize=(10, 3.5))
lo, hi = np.percentile(data, [5, 99.5])
ax.imshow(data, origin="lower", cmap="gray", vmin=lo, vmax=hi, aspect="auto")
ax.set(xlabel="column (dispersion)", ylabel="row", title="a raw science frame: 6 lit traces, 3 dark calibration fibers")
plt.show()

## Ingest

Nothing downstream of ingest sees a raw header. `scan_folder` classifies every frame through
the instrument's header map into a `FrameRef` whose `FrameMeta` speaks the pipeline's
vocabulary; `load_frame` applies the detector's crop and orientation.

In [ ]:
from mastpipe.io.headers import HeaderMap
from mastpipe.io.ingest import load_frame, scan_folder

header_map = HeaderMap.from_dict(load_header_map("highspec"))
refs = scan_folder(night, "highspec", header_map)
pd.DataFrame(
    {
        "source": [r.meta.source for r in refs],
        "type": [r.meta.exposure_type for r in refs],
        "exposure_s": [r.meta.exposure_time for r in refs],
        "mid_time": [r.meta.mid_time for r in refs],
        "rule": [r.meta.classification_rule for r in refs],
    }
).sort_values("mid_time").reset_index(drop=True)

In [ ]:
by_type = {}
for ref in refs:
    by_type.setdefault(ref.meta.exposure_type, []).append(load_frame(ref, effective.detector))
{kind: len(frames) for kind, frames in by_type.items()}

## Stage 1 — clean

Masters are median stacks with validity keys; `build_calibration_set` turns them into what a
frame needs (bias, dark rate, background), and `calibrate_frame` applies the notebook's
arithmetic and builds the variance and mask. Cosmic rays are flagged on science frames with
a median filter that runs along the dispersion axis so emission-line cores are spared.

In [ ]:
from mastpipe.steps.calib import build_calibration_set, build_master, calibrate_frame
from mastpipe.steps.cosmics import flag_cosmic_rays

bias = build_master(by_type["bias"], "bias", params.calib)
dark = build_master(by_type["dark"], "dark", params.calib)
background = build_master(by_type["background"], "background", params.calib)
calib = build_calibration_set(bias, dark, background, effective.detector)

science_raw = by_type["science"][0]
science = calibrate_frame(science_raw, calib, subtract_background=True, saturation=effective.detector.saturation)
science = replace(science, mask=flag_cosmic_rays(science.data, science.variance, science.mask,
                                                   params.calib.cosmic_sigma, params.calib.cosmic_size))
print(f"bias level {bias.data.mean():.1f} ADU, dark rate {np.nanmedian(calib.dark_rate):.3f} ADU/s")
print(f"calibrated frame: {science.data.shape}, {int(science.mask.sum())} masked pixels")

## Stage 2 — traces and extraction

Traces are found on the flat master, matched to the layout table and **re-keyed by the
layout's trace ids** (`layout_indexed`); detections that match no row are dropped. Every
later per-fiber object is keyed by those ids.

In [ ]:
from mastpipe.steps.trace import find_traces, layout_indexed, match_layout, relative_throughput

flats = [calibrate_frame(f, calib, subtract_background=params.calib.flat_subtract_background) for f in by_type["flat"]]
flat = build_master(flats, "flat", params.calib)
detected = find_traces(flat.data, params.trace)
traceset = layout_indexed(match_layout(detected, layout, params.trace.layout_tolerance))
throughput = relative_throughput(flats[0], traceset, params.extract.halfwidth)

print(f"{len(detected)} detected, {len(traceset)} matched, missing {traceset.missing}, extra {traceset.extra}")
pd.DataFrame(
    {"fiber": [t.index for t in traceset], "role": [t.role for t in traceset], "telescope": [t.telescope for t in traceset],
     "y_center": [round(t.y_center, 2) for t in traceset], "rms_px": [round(t.rms, 3) for t in traceset],
     "throughput": [round(throughput[t.index], 3) for t in traceset]}
)

In [ ]:
profile = np.nanmedian(flat.data[:, 700:900], axis=1)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(profile, color="0.3")
for t in traceset:
    ax.axvline(t.y_center, color="tab:red" if t.role == "science" else "tab:blue" if t.role == "sky" else "tab:green", alpha=0.6)
ax.set(xlabel="row", ylabel="flat level (ADU)", title="collapsed flat profile with the matched traces (green arc, red science, blue sky)")
plt.show()

In [ ]:
from mastpipe.steps.extract import extract_aperture

spectra = extract_aperture(science, traceset, params.extract)
fig, ax = plt.subplots(figsize=(10, 3))
for s in spectra:
    if s.telescope == "T01":
        ax.plot(s.pixel, s.flux, lw=0.8, label=f"fiber {s.fiber} ({s.role})")
ax.set(xlabel="pixel", ylabel="ADU", title="raw aperture extraction, telescope T01")
ax.legend()
plt.show()

## Stage 3 — wavelengths

### Lines on the full arc

The full arc is calibrated and extracted like any frame, with a continuum fitted so the
cross-correlation between fibers works on continuum-subtracted flux. `find_lines` then
measures every emission line on the two-dimensional strip around each trace, with flags that
say why a line would or would not be used.

In [ ]:
from mastpipe.steps.extract import continuum_subtract_all
from mastpipe.steps.lines import find_lines, lines_summary

arc_ref = next(r for r in refs if r.meta.exposure_type == "arc" and truth.frames.set_index("file").loc[r.meta.source, "arc_kind"] == "full")
full_arc = calibrate_frame(load_frame(arc_ref, effective.detector), calib)
arc_spectra = continuum_subtract_all(extract_aperture(full_arc, traceset, params.extract), params.extract)
fluxes = [s.flux - s.continuum for s in arc_spectra]
tables, strips = find_lines(full_arc.data, full_arc.variance, traceset, params.lines)
lines_summary(tables)

### The first reference solution

A reference is made once per grating and layout from two anchor pairs (wavelength, pixel)
that a person reads off the anchor panels. Here the anchors come from the simulator's truth.
`solve_full_arc` seeds every other fiber from the anchored one by cross-correlation and then
solves each independently: identify lines through the current model, fit the grating
equation, clip, repeat until the wavelength map stops moving.

In [ ]:
from mastpipe.steps.atlas import catalogue_arrays
from mastpipe.steps.geometry import grating_geometry
from mastpipe.steps.wavecal import build_reference, fiber_agreement, solve_full_arc
from mastpipe.steps.wavemodel import GratingModel, anchor_model

n_pix = full_arc.data.shape[1]
geom = grating_geometry(effective.geometry, effective.detector.pixel_size_um)
cat_wl, cat_sp = catalogue_arrays(line_table, params.wavecal.usable_only)

# two bright, well-separated, on-chip catalogue lines of fiber 0, at their true pixels
bright = np.where(truth.lines_on_chip & (truth.line_logI > 2.0))[0]
i1, i2 = bright[0], bright[-1]
pairs = [(float(truth.lines_nm[i1]), float(truth.line_pixels[0][i1])), (float(truth.lines_nm[i2]), float(truth.line_pixels[0][i2]))]
print("anchors:", [(round(w, 4), round(x, 2)) for w, x in pairs])

model0 = anchor_model(pairs, geom, n_pix)
solutions = solve_full_arc(tables, fluxes, traceset, model0, cat_wl, cat_sp, params.wavecal, ref_fiber=0, geom=geom)
reference = build_reference("highspec", "mgb518", "tutorial9", solutions, created=arc_ref.meta.mid_time,
                            source_checksums=(arc_ref.meta.checksum,))
print("reference", reference.identifier, "valid from", reference.valid_from)
pd.DataFrame({"fiber": list(solutions), "lines": [s.n_lines for s in solutions.values()],
              "rms_px": [round(s.rms_px, 4) for s in solutions.values()]})

In [ ]:
from mastpipe.steps.wavemodel import model_from_solution

truth_model = GratingModel(x_c=truth.x_c, B=truth.B, alpha=geom.alpha_rad, poly=None, n_pix=n_pix, geom=geom)
x = np.arange(n_pix)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for fiber, sol in solutions.items():
    model = model_from_solution(sol, geom)
    axes[0].plot(sol.lines.x, sol.lines.dx_px, ".", label=f"fiber {fiber}")
    axes[1].plot(x, (model.wavelength(x) - truth_model.wavelength(x)) / model.dispersion(x), lw=0.8, label=f"fiber {fiber}")
axes[0].set(xlabel="pixel", ylabel="fit residual (px)", title="identified lines: residuals")
axes[1].set(xlabel="pixel", ylabel="(solved − truth) / dispersion", title="solution against the simulator's truth")
axes[0].legend(fontsize=7, ncol=3)
plt.tight_layout()
plt.show()
fiber_agreement(solutions).head()

### Drift from a calibration-fiber arc

Between full arcs only the calibration fibers see the lamp. For each of them the measured
lines are matched to the reference solution's lines and a linear drift in pixels (shift and
stretch) is fitted; a low-order polynomial in trace position smooths it across the detector.

In [ ]:
from mastpipe.steps.drift import fit_drift

cal_ref = next(r for r in refs if r.meta.exposure_type == "arc" and r is not arc_ref)
cal_arc = calibrate_frame(load_frame(cal_ref, effective.detector), calib)
cal_tables, _ = find_lines(cal_arc.data, cal_arc.variance, traceset, params.lines)
tables_by_fiber = {t.index: tab for t, tab in zip(traceset, cal_tables)}
drift = fit_drift(tables_by_fiber, reference, traceset, cal_ref.meta, params.drift)

planted = truth.frames.set_index("file").loc[cal_ref.meta.source, "shift_px"]
print(f"planted shift {planted:.3f} px; model usable: {drift.usable}, scatter {drift.scatter_px:.4f} px")
drift.per_fiber

### Wavelengths for the science frame

A science frame is paired with calibration-fiber arcs **by time** (`pair_arcs`), and the drift
of the paired arcs is undone before the reference model gives each pixel its wavelength
(`apply_drift`). The frame's status says what happened.

In [ ]:
from mastpipe.pipeline.arcs import pair_arcs, status_for
from mastpipe.steps.drift import apply_drift

pairs_used = pair_arcs(science.meta, [(cal_ref.meta, drift)], reference, params.drift)
wavelengths, skipped = apply_drift(reference, pairs_used, traceset, geom,
                                   tolerance=params.trace.layout_tolerance, n_pix=n_pix)
spectra = [replace(s, wavelength=wavelengths[s.fiber]) if s.fiber in wavelengths else s for s in spectra]
print("status:", status_for(pairs_used), "| arcs paired:", len(pairs_used), "| fibers skipped:", skipped)

## Stage 4 — sky subtraction

Each science fiber minus the sky fiber of the same telescope (from the layout), scaled by the
flat's relative throughput and resampled in wavelength space. The raw extraction stays; the
result is a new version of the spectrum.

In [ ]:
from mastpipe.steps.sky import subtract_sky_all

spectra, reasons = subtract_sky_all(spectra, traceset, throughput, params.sky)
print("not subtracted:", reasons)
s = next(s for s in spectra if s.telescope == "T01" and s.role == "science")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(s.wavelength, s.flux, lw=0.8, color="0.5", label="raw extraction")
ax.plot(s.wavelength, s.sky, lw=0.8, color="tab:blue", label="sky (resampled, scaled)")
ax.plot(s.wavelength, s.versions["skysub"].flux, lw=0.8, color="tab:red", label="sky-subtracted")
ax.set(xlabel="wavelength (nm)", ylabel="ADU", title=f"fiber {s.fiber}, telescope T01, target {s.target}")
ax.legend()
plt.show()

## Combination

The science fibers of one target in the exposure are combined on a common wavelength grid
with inverse-variance weights, every version separately; excluded contributors are listed
with a reason.

In [ ]:
from mastpipe.steps.combine import combine_spectra, group_by_target

groups = group_by_target(spectra)
target, members = next(iter(groups.items()))
combined = combine_spectra(members, target, params.combine)
print(target, "from", len(members), "fibers;", "versions:", list(combined.versions))
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(combined.wavelength, combined.versions["raw"].flux, lw=0.8, color="0.5", label="raw")
ax.plot(combined.wavelength, combined.versions["skysub"].flux, lw=0.8, color="tab:red", label="sky-subtracted")
ax.set(xlabel="wavelength (nm)", ylabel="ADU", title=f"combined spectrum of {target}")
ax.legend()
plt.show()
combined.contributors

## Writing and reading a product

A product is a FITS file with one row per fiber, the raw extraction and every version as
columns, the drift table, and provenance cards. The reader gives back the same dataclasses.

In [ ]:
from mastpipe import __version__
from mastpipe.cli import main
from mastpipe.config.loader import config_hash
from mastpipe.core.model import ExposureSpectra
from mastpipe.io.products import product_name, read_spectra, write_spectra
from mastpipe.pipeline.provenance import Provenance

es = ExposureSpectra(meta=science.meta, spectra=spectra, wavelength_status=status_for(pairs_used),
                     reference_id=reference.identifier, reference_hash=reference.hash,
                     drift_arcs=tuple(sorted(m.checksum for m, _ in [(cal_ref.meta, drift)])))
prov = Provenance(pipeline_version=__version__, config_hash=config_hash(effective),
                  inputs=[science.meta.checksum], calibrations=["bias", "dark", "background", "flat"],
                  wavelength_status=es.wavelength_status, reference_id=es.reference_id,
                  reference_hash=es.reference_hash, drift_arcs=list(es.drift_arcs))
path = work / product_name(science.meta, "spec")
write_spectra(path, es, prov, drift=drift, weights={cal_ref.meta.checksum: 1.0})

back, back_prov, back_drift = read_spectra(path)
print(path.name, "|", len(back.spectra), "fibers |", back.wavelength_status, "|", sorted(back.spectra[1].versions))
main(["show", str(path)])

## Where to go next

- `mastpipe quicklook` runs exactly this chain for every frame of a folder, with the
  calibration library, idempotence and QC files; [Inspecting a night's products](inspecting-products.ipynb)
  reads what it writes.
- The functions used here are documented in the [API reference](../api/index.md); their
  methods and the choices behind them in [How it works](../how-it-works/index.md).
- To run on a real night, replace the simulated folder with yours and the hand-made layout
  with the instrument's preset (`load_layout("highspec", "survey45")`).